In [1]:
import json
import os
import random
import time
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image, ImageDraw, ImageFont
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights, resnet50, vgg16
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign

In [2]:
###########################################
# 3. Classe MultiTaskModel (Detecção + Atributos Globais)
###########################################
class MultiTaskModel(nn.Module):
    def __init__(self, detection_model, backbone, num_weather, num_scene, num_time, in_features):
        super(MultiTaskModel, self).__init__()
        self.detection_model = detection_model
        self.backbone = backbone
        self.attr_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc_weather = nn.Linear(in_features, num_weather)
        self.fc_scene = nn.Linear(in_features, num_scene)
        self.fc_timeofday = nn.Linear(in_features, num_time)
    def forward(self, images, targets=None, global_attrs=None):
        # Em treinamento, retorna losses; em avaliação, retorna as detecções
        if self.training:
            detection_loss = self.detection_model(images, targets)
        else:
            detection_loss = self.detection_model(images)
        imgs_tensor = torch.stack(images)
        feats = self.backbone(imgs_tensor)
        pooled = self.attr_pool(feats)
        pooled = pooled.view(pooled.size(0), -1)
        weather_logits = self.fc_weather(pooled)
        scene_logits = self.fc_scene(pooled)
        timeofday_logits = self.fc_timeofday(pooled)
        if self.training and global_attrs is not None:
            weather_labels = torch.stack([attr["weather"] for attr in global_attrs]).to(images[0].device)
            scene_labels = torch.stack([attr["scene"] for attr in global_attrs]).to(images[0].device)
            timeofday_labels = torch.stack([attr["timeofday"] for attr in global_attrs]).to(images[0].device)
            loss_weather = nn.functional.cross_entropy(weather_logits, weather_labels)
            loss_scene = nn.functional.cross_entropy(scene_logits, scene_labels)
            loss_timeofday = nn.functional.cross_entropy(timeofday_logits, timeofday_labels)
            attr_loss = loss_weather + loss_scene + loss_timeofday
        else:
            attr_loss = 0
        return detection_loss, attr_loss, weather_logits, scene_logits, timeofday_logits

In [3]:
# ========= Configurações =========
model_path = "multi_task_model_mobilenet_20.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([transforms.ToTensor()])

model_type = "mobilenet"  # Altere conforme necessário

# ========= Load mappings =========
with open("helpers/categories.json", "r") as f:
    category_to_label = json.load(f)
with open("helpers/weather.json", "r") as f:
    weather_to_label = json.load(f)
with open("helpers/scene.json", "r") as f:
    scene_to_label = json.load(f)
with open("helpers/timeofday.json", "r") as f:
    timeofday_to_label = json.load(f)

# ========= Carrega modelo completo =========
print(f"📦 Carregando modelo de '{model_path}'...")
multi_task_model = torch.load(model_path, map_location=device, weights_only=False)
multi_task_model.eval()
print("✅ Modelo carregado.")

📦 Carregando modelo de 'multi_task_model_mobilenet_20.pth'...
✅ Modelo carregado.


In [4]:
###########################################
# 6. Inferência em múltiplas imagens do conjunto de teste
###########################################
def invert_mapping(mapping):
    return {v: k for k, v in mapping.items()}


def infer_and_annotate_image(image_path, model, device, transform,
                             category_to_label, weather_to_label,
                             scene_to_label, timeofday_to_label,
                             confidence_threshold=0.5):
    inv_category = invert_mapping(category_to_label)
    inv_weather = invert_mapping(weather_to_label)
    inv_scene = invert_mapping(scene_to_label)
    inv_timeofday = invert_mapping(timeofday_to_label)
    orig_image = Image.open(image_path).convert("RGB")
    input_image = transform(orig_image).to(device).unsqueeze(0)
    model.eval()
    with torch.no_grad():
        detections = model.detection_model([input_image.squeeze(0)])
        feats = model.backbone(input_image)
        pooled = model.attr_pool(feats)
        pooled = pooled.view(pooled.size(0), -1)
        weather_logits = model.fc_weather(pooled)
        scene_logits = model.fc_scene(pooled)
        timeofday_logits = model.fc_timeofday(pooled)
        weather_pred = int(torch.argmax(weather_logits, dim=1).item())
        scene_pred = int(torch.argmax(scene_logits, dim=1).item())
        timeofday_pred = int(torch.argmax(timeofday_logits, dim=1).item())
        global_attributes = {
            "weather": inv_weather.get(weather_pred, str(weather_pred)),
            "scene": inv_scene.get(scene_pred, str(scene_pred)),
            "timeofday": inv_timeofday.get(timeofday_pred, str(timeofday_pred))
        }
        detection = detections[0]
        boxes = detection["boxes"].cpu().numpy().tolist()
        labels = detection["labels"].cpu().numpy().tolist()
        scores = detection["scores"].cpu().numpy().tolist()
        detection_list = []
        for bbox, label, score in zip(boxes, labels, scores):
            if score < confidence_threshold:
                continue
            detection_list.append({
                "category": inv_category.get(label, str(label)),
                "score": score,
                "box": bbox
            })
    draw = ImageDraw.Draw(orig_image)
    try:
        font = ImageFont.truetype("arial.ttf", 15)
    except Exception:
        font = ImageFont.load_default()
    for det in detection_list:
        bbox = det["box"]
        text = f"{det['category']}: {det['score']:.2f}"
        draw.rectangle(bbox, outline="red", width=2)
        draw.text((bbox[0], bbox[1] - 10), text, fill="red", font=font)
    attr_text = f"Weather: {global_attributes['weather']}, Scene: {global_attributes['scene']}, Time: {global_attributes['timeofday']}"
    draw.text((10, 10), attr_text, fill="blue", font=font)
    return orig_image, {"global_attributes": global_attributes, "detections": detection_list}


def infer_on_test_folder(test_folder, output_folder, model, device, transform,
                         category_to_label, weather_to_label, scene_to_label, timeofday_to_label,
                         confidence_threshold=0.5):
    os.makedirs(output_folder, exist_ok=True)
    results = {}
    for filename in os.listdir(test_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_path = os.path.join(test_folder, filename)
            annotated_img, output = infer_and_annotate_image(image_path, model, device, transform,
                                                             category_to_label, weather_to_label, scene_to_label,
                                                             timeofday_to_label,
                                                             confidence_threshold)
            base_name = os.path.splitext(filename)[0]
            annotated_image_path = os.path.join(output_folder, f"{base_name}_annotated.jpg")
            json_output_path = os.path.join(output_folder, f"{base_name}.json")
            annotated_img.save(annotated_image_path)
            with open(json_output_path, "w") as f:
                json.dump(output, f, indent=2)
            print(f"Processado: {filename}")
            results[filename] = output
    return results


# ========= Executar inferência =========
print("🔎 Iniciando inferência no conjunto de teste...")
test_folder = "images/test"
output_folder = f"{model_type}_outputs"
inference_results = infer_on_test_folder(test_folder, output_folder, multi_task_model, device, transform,
                                         category_to_label, weather_to_label, scene_to_label, timeofday_to_label,
                                         confidence_threshold=0.5)

# ========= Exibir resultados resumidos =========
print("📊 Inferência concluída. Exemplos de resultados:")
for img_name, result in list(inference_results.items())[:10]:
    print(f"\n🖼️ {img_name}")
    print(json.dumps(result, indent=2))

🔎 Iniciando inferência no conjunto de teste...
Processado: cabe1040-5f02711e.jpg
Processado: cac07407-951977c8.jpg
Processado: cac07407-ba37148a.jpg
Processado: cac07407-e969f06a.jpg
Processado: cad4a646-68a8d136.jpg
Processado: cad7fdff-d9946f73.jpg
Processado: cadd00a0-efef6501.jpg
Processado: cae6c88d-0db9c3ab.jpg
Processado: caf8496d-6ce3dbda.jpg
Processado: caf92153-cfcdf92a.jpg
Processado: cafb3e6f-22f2007e.jpg
Processado: cafbae4e-1d69e770.jpg
Processado: cafbb9cb-c8d8b9df.jpg
Processado: caff2200-238a0388.jpg
Processado: caff5d9a-35754dcf.jpg
Processado: cb0ad251-7f86477e.jpg
Processado: cb146e98-77442601.jpg
Processado: cb14e978-6c4d53f3.jpg
Processado: cb14e978-a5eb5afb.jpg
Processado: cb16478b-2c1ed614.jpg
Processado: cb25d8ec-e52ef234.jpg
Processado: cb2fe290-7246a85f.jpg
Processado: cb32ab44-e8b00418.jpg
Processado: cb37bfff-1b87d685.jpg
Processado: cb3f27ee-b8c0119f.jpg
Processado: cb421747-f06aec7d.jpg
Processado: cb58f48b-957bfd2c.jpg
Processado: cb63ddeb-c7aee033.jpg
P

KeyboardInterrupt: 